<a href="https://colab.research.google.com/github/chanceCoderByPassion/DL_CV/blob/master/Yolo_soccer_distance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.5 MB/s eta 0:00:00


In [3]:

import cv2
import numpy as np
from ultralytics import YOLO

# --- CONFIGURATION ---
# The scale factor: how many pixels per meter on your "virtual" pitch
# This is a simplified approach for pan/tilt; full 3D reconstruction is a higher level.
PIXELS_PER_METER = 20

model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture("/content/drive/My Drive/soccer_match_panning_1.mp4")

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_size = (width, height)



import cv2
import os

# Define the directory on your Drive
output_dir = '/content/drive/My Drive/Soccer_Analysis/'

# Create the folder if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'processed_match.mp4')

# Initialize VideoWriter
# Use 'avc1' or 'mp4v' for the codec
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = 25.0 # Match this to your input video
#frame_size = (640, 360) # Match this to your input frame size

out = cv2.VideoWriter(output_path, fourcc, fps, frame_size)






# Feature tracking params (ORB or SIFT)
orb = cv2.ORB_create(nfeatures=1000)
matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

# State variables
prev_frame_gray = None
prev_kps, prev_des = None, None
cumulative_h = np.eye(3) # Identity matrix to start
tracker_history = {} # {id: {"last_real_pos": (x, y), "total_dist": 0}}

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("video could not be opened")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    kps, des = orb.detectAndCompute(gray, None)

    # 1. CAMERA MOTION ESTIMATION
    if prev_frame_gray is not None and des is not None:
        # Match features between current and previous frame
        matches = matcher.match(prev_des, des)
        matches = sorted(matches, key=lambda x: x.distance)[:100]

        src_pts = np.float32([prev_kps[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
        dst_pts = np.float32([kps[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

        # Find the Homography between frames (The Camera Movement)
        m_matrix, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

        if m_matrix is not None:
            # Accumulate movement to keep a global coordinate system
            cumulative_h = cumulative_h @ m_matrix

    # 2. YOLO TRACKING
    results = model.track(frame,tracker="botsort.yaml", persist=True, classes=[0], verbose=False)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            # Feet position in current frame pixels
            px, py = (box[0] + box[2]) / 2, box[3]

            # 3. TRANSFORM PIXELS TO GLOBAL "WORLD" COORDINATES
            # We use the inverse of the cumulative movement to find the "fixed" pitch position
            point = np.array([[px, py]], dtype="float32").reshape(-1, 1, 2)
            world_pos = cv2.perspectiveTransform(point, np.linalg.inv(cumulative_h))[0][0]

            real_x, real_y = world_pos[0] / PIXELS_PER_METER, world_pos[1] / PIXELS_PER_METER

            if track_id not in tracker_history:
                tracker_history[track_id] = {"last_pos": (real_x, real_y), "total_dist": 0.0}
            else:
                prev_x, prev_y = tracker_history[track_id]["last_pos"]
                # Distance calculation
                step_dist = np.sqrt((real_x - prev_x)**2 + (real_y - prev_y)**2)

                # Filter out small jitter and tracking noise
                if 0.05 < step_dist < 2.0:
                    tracker_history[track_id]["total_dist"] += step_dist
                    tracker_history[track_id]["last_pos"] = (real_x, real_y)

            # Draw Results
            dist = tracker_history[track_id]["total_dist"]
            cv2.putText(frame, f"ID{track_id}: {dist:.1f}m", (int(box[0]), int(box[1]-10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 255, 0), 2)

    # Update frame state
    prev_frame_gray = gray
    prev_kps, prev_des = kps, des

    #cv2_imshow(frame)
    out.write(frame)
    #if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
out.release()
#cv2.destroyAllWindows()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 224ms
Prepared 1 package in 58ms
Installed 1 package in 4ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

video could not be opened
